# MDR-TS v5.2
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Thu Jan 15th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v5.2
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_2.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**
- Reproducing the same experiment as in v5.1 but starting from the bottom and moving up (wrt to number of features).
- The results should be the same as in v5.1. 

For other info see `MDR-TS-v5.1`

## 0. Imports

In [21]:
import os
import json
import math
import random
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML / Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Gradient Boosting (baseline model)
import xgboost as xgb
from xgboost import XGBRegressor

# PyTorch
import torch

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Utilities
from itertools import product
from tqdm.auto import tqdm
import re

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

F32_MAX = np.finfo(np.float32).max

imports loaded
using: cpu


## 1. Environment Setup

In [2]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [3]:
# Project paths
VERSION = "v5"
SUBVERSION = "v5.2"
RUN_NAME = "mdr_ts_v5_2"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR/"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR/
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR//Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR//Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR//Models/Temporal/v5/v5.2

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [4]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_2.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_2.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_2.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

DROP_COLS = ["slope", "elev"]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=cols_present, inplace=True)
    print(f"{name}: dropped columns {cols_present}")
    print(f"\n{name}: shape={d.shape}")
    print(f"{name}: columns={len(d.columns)}")

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_all/train_derived_all.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_all/val_derived_all.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_all/test_derived_all.csv
train: dropped columns ['slope', 'elev']

train: shape=(15904, 352)
train: columns=352
val: dropped columns ['slope', 'elev']

val: shape=(3408, 352)
val: columns=352
test: dropped columns ['slope', 'elev']

test: shape=(3408, 352)
test: columns=352


In [5]:
print("Common columns across splits:",
      (set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: {'V_rollstd_E_SAR_diff_kobs14', 'C_lag_F_NDMI_kobs6', 'A_pct_s2_b12', 'A_grad_G_API_kobs30', 'G_API', 'C_lag_F_NDVI_kobs30', 'V_rollmin_E_SAR_diff_kobs14', 'D_fft_ent_E_SAR_ratio_kobs30', 'E_rough_s1_vh_kobs7', 'V_rollrng_G_API_kobs7', 'C_lag_E_SAR_ratio_kobs5', 'V_ema_G_API_kobs7', 'C_lag_F_NDMI_kobs30', 'V_rollmean_E_SAR_diff_kobs7', 'C_lag_LST_modis_kobs12', 'V_rollmax_F_NDMI_kobs7', 'A_d_F_NDMI_kobs30', 'V_rollmean_s2_b12_kobs30', 'V_rollrng_s2_b11_kobs7', 'V_rollmax_E_SAR_ratio_kobs14', 'V_rollstd_s2_b11_kobs14', 'V_rollmean_F_NDMI_kobs30', 'F_NDMI', 'V_rollrng_s2_b12_kobs7', 'V_rollrng_s2_b12_kobs14', 'DOY', 'V_rollmax_LST_modis_kobs14', 'A_pct_G_API', 'C_lag_G_API_kobs2', 'V_rollmax_F_NDVI_kobs7', 'C_lag_E_SAR_ratio_kobs1', 'A_grad_LST_modis_kobs14', 'A_d_E_SAR_diff_kobs14', 'A_d_E_SAR_ratio_kobs7', 's1_vh', 'A_d_s2_b12_kobs2', 'G_DSLR', 'A_d_F_NDMI_kobs5', 'A_pct_E_SAR_ratio', 'A_d_E_SAR_ratio_kobs1', 'A_d_F_NDMI_kobs2', 'V_rollstd_E_SAR_ratio_kobs

## 4. Data Sanity Checks

In [42]:
# Column definitions (baseline)
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

RAW_FEATURES = [
    "longitude",
    "latitude",
    "precip_mm",
    "s1_vv",
    "s1_vh",
    "s2_b4",
    "s2_b8",
    "s2_b11",
    "s2_b12",
    "LST_modis",
    "aspect",
    "DOY",
]

FAM_PREFIX = {
    "raw+A": ("A_",),
    "raw+B": ("V_",),   # B-family is V_*
    "raw+C": ("C_",),
    "raw+D": ("D_",),
    "raw+E": ("E_",),
    "raw+F": ("F_",),
    "raw+G": ("G_",),
    "raw+H": ("H_",),
    "raw+I": ("I_",),
}

FAMILY_PREFIXES = ("A_", "V_", "C_", "D_", "E_", "F_", "G_", "H_", "I_")

def build_feature_families(df):
    cols = list(df.columns)

    # basic exclusions
    cols = [c for c in cols if c not in KEEP_META_COLS and c != TARGET_COL]

    # raw/base features = not derived prefixes
    raw = [c for c in cols if not c.startswith(FAMILY_PREFIXES) and not c.startswith("H_")]

    # families by prefix
    fam = {}
    fam["A"] = [c for c in cols if c.startswith("A_")]
    fam["B"] = [c for c in cols if c.startswith("V_")]     # B is V_*
    fam["C"] = [c for c in cols if c.startswith("C_")]
    fam["D"] = [c for c in cols if c.startswith("D_")]     # includes anomalies + fft named as D_*
    fam["E"] = [c for c in cols if c.startswith("E_")]
    fam["F"] = [c for c in cols if c.startswith("F_")]
    fam["G"] = [c for c in cols if c.startswith("G_")]
    fam["H"] = [c for c in cols if c.startswith("H_")]     # coupling names start with H_
    fam["I"] = [c for c in cols if c.startswith("I_")]

    return raw, fam

In [7]:
raw, fam = build_feature_families(train_df)

feature_sets = {
    "raw_only": raw,
    "raw+A": raw + fam["A"],
    "raw+B": raw + fam["B"],
    "raw+C": raw + fam["C"],
    "raw+D": raw + fam["D"],
    "raw+E": raw + fam["E"],
    "raw+F": raw + fam["F"],
    "raw+G": raw + fam["G"],
    "raw+H": raw + fam["H"],
    "raw+I": raw + fam["I"],
}

## 5. Train / Validation / Test Split

In [ ]:
def make_family_splits(train_df, val_df, test_df, feature_sets, target_col=TARGET_COL):
    splits = {}

    for name, cols in feature_sets.items():
        cols = [c for c in cols if c not in KEEP_META_COLS and c != target_col]

        missing_train = set(cols) - set(train_df.columns)
        missing_val   = set(cols) - set(val_df.columns)
        missing_test  = set(cols) - set(test_df.columns)
        missing = missing_train | missing_val | missing_test
        if missing:
            raise ValueError(f"[{name}] Missing columns in at least one split: {sorted(missing)[:30]}")

        X_train = train_df[cols].astype(np.float32)
        X_val   = val_df[cols].astype(np.float32)
        X_test  = test_df[cols].astype(np.float32)

        y_train = train_df[target_col].astype(np.float32)
        y_val   = val_df[target_col].astype(np.float32)
        y_test  = test_df[target_col].astype(np.float32)

        splits[name] = dict(
            features=cols,
            X_train=X_train, X_val=X_val, X_test=X_test,
            y_train=y_train, y_val=y_val, y_test=y_test
        )

    return splits

In [10]:
splits_by_family = make_family_splits(train_df, val_df, test_df, feature_sets)

In [11]:
for name, pack in splits_by_family.items():
    print(f"{name:7s} | n_feat={len(pack['features']):3d} | "
          f"train={pack['X_train'].shape} val={pack['X_val'].shape} test={pack['X_test'].shape}")

raw_only | n_feat= 10 | train=(15904, 10) val=(3408, 10) test=(3408, 10)
raw+A   | n_feat= 90 | train=(15904, 90) val=(3408, 90) test=(3408, 90)
raw+B   | n_feat=178 | train=(15904, 178) val=(3408, 178) test=(3408, 178)
raw+C   | n_feat= 66 | train=(15904, 66) val=(3408, 66) test=(3408, 66)
raw+D   | n_feat= 22 | train=(15904, 22) val=(3408, 22) test=(3408, 22)
raw+E   | n_feat= 17 | train=(15904, 17) val=(3408, 17) test=(3408, 17)
raw+F   | n_feat= 13 | train=(15904, 13) val=(3408, 13) test=(3408, 13)
raw+G   | n_feat= 16 | train=(15904, 16) val=(3408, 16) test=(3408, 16)
raw+H   | n_feat= 14 | train=(15904, 14) val=(3408, 14) test=(3408, 14)
raw+I   | n_feat= 11 | train=(15904, 11) val=(3408, 11) test=(3408, 11)


## 6. Model Definition

In [46]:
def _sanitize_X_for_xgb(X: pd.DataFrame) -> pd.DataFrame:
    X = X.astype(np.float32, copy=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_MAX, np.nan)
    return X

def train_xgb_family(
    pack: dict,
    *,
    family_name: str,  # <-- NEW: pass the key like "raw+A"
    max_depth: int = 3,
    min_child_weight: float = 5,
    reg_lambda: float = 30,
    subsample: float = 1.0,
    colsample_bytree: float = 0.6,
    learning_rate: float = 0.05,
    num_boost_round: int = 6000,
    early_stopping_rounds: int = 75,
    random_state: int = 42,
    importance_type: str = "gain",
    top_k_importance: int = 5,) -> dict:

    X_train, y_train = pack["X_train"], pack["y_train"]
    X_val, y_val     = pack["X_val"], pack["y_val"]
    X_test, y_test   = pack["X_test"], pack["y_test"]
    feat_names       = list(pack["features"])

    # sanitize
    X_train = _sanitize_X_for_xgb(X_train)
    X_val   = _sanitize_X_for_xgb(X_val)
    X_test  = _sanitize_X_for_xgb(X_test)

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feat_names, missing=np.nan)
    dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=feat_names, missing=np.nan)
    dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=feat_names, missing=np.nan)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "eta": learning_rate,
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "lambda": reg_lambda,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "seed": random_state,
        "verbosity": 0,
        "tree_method": "hist",
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False,
    )

    best_iter = getattr(booster, "best_iteration", None)
    best_ntree_limit = getattr(booster, "best_ntree_limit", None)

    def _pred(dm):
        if best_ntree_limit is not None and best_ntree_limit > 0:
            return booster.predict(dm, ntree_limit=best_ntree_limit)
        return booster.predict(dm)

    yhat_tr = _pred(dtrain)
    yhat_va = _pred(dval)
    yhat_te = _pred(dtest)

    def _metrics(y, yhat):
        rmse = float(np.sqrt(mean_squared_error(y, yhat)))
        return {
            "R2": float(r2_score(y, yhat)),
            "MAE": float(mean_absolute_error(y, yhat)),
            "RMSE": rmse,
        }

    metrics = {
        "train": _metrics(y_train, yhat_tr),
        "val":   _metrics(y_val,   yhat_va),
        "test":  _metrics(y_test,  yhat_te),
    }

    # Full importance
    score = booster.get_score(importance_type=importance_type)  # keys are feature names
    imp = np.array([float(score.get(f, 0.0)) for f in feat_names], dtype=float)

    imp_df = pd.DataFrame({
        "feature": feat_names,
        importance_type: imp,
    }).sort_values(by=importance_type, ascending=False, kind="mergesort").reset_index(drop=True)

    # Family-only top-k for reporting
    if family_name == "raw_only":
        report_df = imp_df.copy()
    else:
        prefixes = FAM_PREFIX.get(family_name, None)
        if prefixes is None:
            raise ValueError(f"Unknown family_name '{family_name}'. Add it to FAM_PREFIX.")
        report_df = imp_df[imp_df["feature"].str.startswith(prefixes)].copy()

    top_df = report_df.sort_values(by=importance_type, ascending=False).head(top_k_importance).reset_index(drop=True)

    return {
        "booster": booster,
        "best_iteration": best_iter,
        "best_ntree_limit": best_ntree_limit,
        "metrics": metrics,
        "importance_df": imp_df,
        "top_importance_df": top_df,
    }

In [47]:
def run_family_experiments(
    splits_by_family: dict,
    *,
    max_depth: int = 3,
    min_child_weight: float = 5,
    reg_lambda: float = 30,
    subsample: float = 1.0,
    colsample_bytree: float = 0.6,
    learning_rate: float = 0.05,
    num_boost_round: int = 6000,
    early_stopping_rounds: int = 75,
    random_state: int = 42,
    importance_type: str = "gain",
    top_k_importance: int = 5):

    results = {}
    metric_rows = []
    top_rows = []

    baseline_test_r2 = None

    ordered_keys = list(splits_by_family.keys())
    if "raw_only" in ordered_keys:
        ordered_keys = ["raw_only"] + [k for k in ordered_keys if k != "raw_only"]

    for name in ordered_keys:
        pack = splits_by_family[name]

        out = train_xgb_family(
            pack,
            family_name=name,
            max_depth=max_depth,
            min_child_weight=min_child_weight,
            reg_lambda=reg_lambda,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            learning_rate=learning_rate,
            num_boost_round=num_boost_round,
            early_stopping_rounds=early_stopping_rounds,
            random_state=random_state,
            importance_type=importance_type,
            top_k_importance=top_k_importance,
        )
        results[name] = out

        if name == "raw_only":
            baseline_test_r2 = out["metrics"]["test"]["R2"]

        row = {
            "name": name,
            "n_feat": len(pack["features"]),
            "best_iter": out["best_iteration"],
            "train_R2": out["metrics"]["train"]["R2"],
            "val_R2":   out["metrics"]["val"]["R2"],
            "test_R2":  out["metrics"]["test"]["R2"],
            "test_MAE": out["metrics"]["test"]["MAE"],
            "test_RMSE": out["metrics"]["test"]["RMSE"],
        }
        if baseline_test_r2 is not None and name != "raw_only":
            row["test_R2_uplift_vs_raw"] = row["test_R2"] - baseline_test_r2
        else:
            row["test_R2_uplift_vs_raw"] = np.nan

        metric_rows.append(row)

        top_df = out["top_importance_df"].copy()
        top_df = top_df.sort_values(by=importance_type, ascending=False).reset_index(drop=True)
        top_df.insert(0, "name", name)
        top_df.insert(1, "rank", np.arange(1, len(top_df) + 1))
        top_rows.append(top_df)

    metrics_df = pd.DataFrame(metric_rows).sort_values("test_R2", ascending=False).reset_index(drop=True)

    top5_df = pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame()
    if not top5_df.empty:
        top5_df = top5_df.sort_values(["name", importance_type], ascending=[True, False]).reset_index(drop=True)

    return metrics_df, top5_df, results

In [48]:
metrics_df, top5_df, results = run_family_experiments(
    splits_by_family,
    max_depth=3,
    min_child_weight=5,
    reg_lambda=30,
    subsample=1.0,
    colsample_bytree=0.6,
    learning_rate=0.05,
    num_boost_round=6000,
    early_stopping_rounds=75,
    importance_type="gain",
    top_k_importance=5,
)

In [49]:
display(metrics_df)

,name,n_feat,best_iter,train_R2,val_R2,test_R2,test_MAE,test_RMSE,test_R2_uplift_vs_raw
0,raw+C,66,180,0.891193,0.796591,0.681632,0.040118,0.052463,0.079456
1,raw+A,90,508,0.896721,0.763101,0.667863,0.041556,0.053586,0.065687
2,raw+B,178,182,0.898275,0.774850,0.666092,0.041196,0.053728,0.063916
3,raw+G,16,614,0.905399,0.798573,0.659174,0.041240,0.054282,0.056998
4,raw+E,17,153,0.809768,0.692876,0.610129,0.044448,0.058056,0.007953
5,raw+F,13,169,0.813300,0.712534,0.609532,0.044106,0.058101,0.007356
6,raw_only,10,222,0.818618,0.708935,0.602176,0.044564,0.058646,NaN
7,raw+D,22,173,0.819589,0.697613,0.600000,0.045488,0.058806,-0.002176
8,raw+I,11,234,0.821392,0.705402,0.597486,0.045397,0.058990,-0.004690
9,raw+H,14,379,0.833990,0.707535,0.594280,0.044992,0.059225,-0.007895


In [50]:
rawA = (
    top5_df
    .loc[top5_df["name"] == "raw+A"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawA)

,name,rank,feature,gain
0,raw+A,1,A_d_G_API_kobs1,0.546162
1,raw+A,2,A_grad_LST_modis_kobs7,0.483721
2,raw+A,3,A_d_G_API_kobs2,0.448454
3,raw+A,4,A_d_G_API_kobs30,0.317390
4,raw+A,5,A_grad_s2_b11_kobs30,0.232089


In [51]:
rawB = (
    top5_df
    .loc[top5_df["name"] == "raw+B"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawB)

,name,rank,feature,gain
0,raw+B,1,V_ema_LST_modis_kobs30,10.807203
1,raw+B,2,V_rollmin_G_API_kobs14,7.331640
2,raw+B,3,V_rollmin_LST_modis_kobs30,7.220735
3,raw+B,4,V_ema_G_API_kobs14,4.141772
4,raw+B,5,V_rollmean_G_API_kobs7,2.440078


In [52]:
rawC = (
    top5_df
    .loc[top5_df["name"] == "raw+C"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawC)

,name,rank,feature,gain
0,raw+C,1,C_smm_G_API_alpha0.85_n5,6.297982
1,raw+C,2,C_lag_LST_modis_kobs12,4.748018
2,raw+C,3,C_lag_LST_modis_kobs30,2.322512
3,raw+C,4,C_lag_G_API_kobs6,1.539671
4,raw+C,5,C_smm_F_NDMI_alpha0.85_n5,1.419127


In [53]:
rawD = (
    top5_df
    .loc[top5_df["name"] == "raw+D"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawD)

,name,rank,feature,gain
0,raw+D,1,D_sa_F_NDMI,0.431063
1,raw+D,2,D_z_F_NDMI,0.416928
2,raw+D,3,D_sa_E_SAR_ratio,0.233489
3,raw+D,4,D_z_E_SAR_ratio,0.195131
4,raw+D,5,D_fft_ent_E_SAR_ratio_kobs30,0.136506


In [54]:
rawE = (
    top5_df
    .loc[top5_df["name"] == "raw+E"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawE)

,name,rank,feature,gain
0,raw+E,1,E_SAR_diff,0.509180
1,raw+E,2,E_rough_s1_vh_kobs7,0.233583
2,raw+E,3,E_SAR_ratio,0.117198
3,raw+E,4,E_rough_s1_vh_kobs14,0.113681
4,raw+E,5,E_rough_s1_vv_kobs7,0.098404


In [55]:
rawF = (
    top5_df
    .loc[top5_df["name"] == "raw+F"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawF)

,name,rank,feature,gain
0,raw+F,1,F_MSI,0.444096
1,raw+F,2,F_NDMI,0.322963
2,raw+F,3,F_NDVI,0.129457


In [56]:
rawG = (
    top5_df
    .loc[top5_df["name"] == "raw+G"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawG)

,name,rank,feature,gain
0,raw+G,1,G_API,1.531489
1,raw+G,2,G_rain_sum_30d,0.357107
2,raw+G,3,G_rain_sum_3d,0.131231
3,raw+G,4,G_rain_sum_7d,0.101207
4,raw+G,5,G_DSLR,0.048658


In [57]:
rawH = (
    top5_df
    .loc[top5_df["name"] == "raw+H"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawH)

,name,rank,feature,gain
0,raw+H,1,H_corr_E_SAR_ratio__F_NDMI_kobs14,0.074187
1,raw+H,2,H_corr_LST_modis__F_NDMI_kobs14,0.047383
2,raw+H,3,H_corr_E_SAR_ratio__F_NDMI_kobs7,0.035400
3,raw+H,4,H_corr_LST_modis__F_NDMI_kobs7,0.021081


In [58]:
rawI = (
    top5_df
    .loc[top5_df["name"] == "raw+I"]
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

display(rawI)

,name,rank,feature,gain
0,raw+I,1,I_ts_spike_s1_vv,0.09586


In [59]:
# NOTE: gain is in terms of MSE reduction
display(top5_df.sort_values(["gain"], ascending=[False]).reset_index(drop=True))

,name,rank,feature,gain
0,raw+B,1,V_ema_LST_modis_kobs30,10.807203
1,raw+B,2,V_rollmin_G_API_kobs14,7.331640
2,raw+B,3,V_rollmin_LST_modis_kobs30,7.220735
3,raw+C,1,C_smm_G_API_alpha0.85_n5,6.297982
4,raw+C,2,C_lag_LST_modis_kobs12,4.748018
5,raw+B,4,V_ema_G_API_kobs14,4.141772
6,raw+B,5,V_rollmean_G_API_kobs7,2.440078
7,raw+C,3,C_lag_LST_modis_kobs30,2.322512
8,raw_only,1,DOY,1.851195
9,raw+C,4,C_lag_G_API_kobs6,1.539671


In [62]:
def build_family_only_importance(results, metrics_df, importance_col="gain"):
    rows = []
    uplift_map = dict(zip(metrics_df["name"], metrics_df["test_R2_uplift_vs_raw"]))

    for name, out in results.items():
        if name == "raw_only" or not name.startswith("raw+"):
            continue
        prefixes = FAM_PREFIX.get(name)
        if prefixes is None:
            continue

        imp = out["importance_df"].copy()
        imp = imp[imp["feature"].str.startswith(prefixes)].copy()
        imp["name"] = name
        imp["uplift"] = float(uplift_map.get(name, 0.0))
        rows.append(imp[["name", "feature", importance_col, "uplift"]])

    fam_imp = pd.concat(rows, ignore_index=True)
    return fam_imp

fam_imp = build_family_only_importance(results, metrics_df, importance_col="gain")

In [83]:
def select_top_features(fam_imp, k=40, importance_col="gain"):
    df = fam_imp.copy()
    df["uplift_pos"] = df["uplift"].clip(lower=0.0)
    df["score"] = df[importance_col] * df["uplift_pos"]

    top = df.sort_values("score", ascending=False).drop_duplicates("feature").head(k)
    return top.reset_index(drop=True)

top40 = select_top_features(fam_imp, k=40, importance_col="gain")
features = top40["feature"].tolist()

print(features)

['V_ema_LST_modis_kobs30', 'C_smm_G_API_alpha0.85_n5', 'V_rollmin_G_API_kobs14', 'V_rollmin_LST_modis_kobs30', 'C_lag_LST_modis_kobs12', 'V_ema_G_API_kobs14', 'C_lag_LST_modis_kobs30', 'V_rollmean_G_API_kobs7', 'V_rollmin_G_API_kobs30', 'V_ema_LST_modis_kobs14', 'V_ema_G_API_kobs7', 'V_rollmax_G_API_kobs14', 'C_lag_G_API_kobs6', 'C_smm_F_NDMI_alpha0.85_n5', 'C_lag_LST_modis_kobs6', 'C_lag_G_API_kobs1', 'G_API', 'V_rollmin_F_NDMI_kobs7', 'C_lag_G_API_kobs5', 'V_rollmax_G_API_kobs7', 'C_lag_G_API_kobs12', 'V_rollmin_E_SAR_diff_kobs30', 'C_lag_G_API_kobs2', 'V_rollmin_F_NDMI_kobs14', 'V_ema_F_NDMI_kobs30', 'V_rollcv_F_NDMI_kobs30', 'V_rollmin_E_SAR_diff_kobs14', 'V_rollmin_F_NDMI_kobs30', 'C_lag_F_NDMI_kobs1', 'C_smm_E_SAR_ratio_alpha0.85_n5', 'C_lag_F_NDMI_kobs2', 'C_smm_E_SAR_diff_alpha0.85_n5', 'V_rollmin_G_API_kobs7', 'V_rollmax_E_SAR_diff_kobs14', 'C_lag_E_SAR_diff_kobs2', 'C_lag_s2_b11_kobs6', 'A_d_G_API_kobs1', 'V_ema_G_API_kobs30', 'C_lag_F_NDMI_kobs12', 'V_rollmax_F_NDMI_kobs14']

## Final Model

In [76]:
X_train = _sanitize_X_for_xgb(train_df[features])
y_train = train_df[TARGET_COL]

X_val   = _sanitize_X_for_xgb(val_df[features])
y_val   = val_df[TARGET_COL]

X_test  = _sanitize_X_for_xgb(test_df[features])
y_test  = test_df[TARGET_COL]

In [77]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features, missing=np.nan)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=features, missing=np.nan)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=features, missing=np.nan)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "eta": 0.05,
    "max_depth": 3,
    "min_child_weight": 5,
    "lambda": 30,
    "subsample": 1.0,
    "colsample_bytree": 0.6,
    "seed": 42,
    "verbosity": 0,
    "tree_method": "hist",
}

booster_final = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=6000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=75,
    verbose_eval=False,
)

In [81]:
best_ntree_limit = getattr(booster_final, "best_ntree_limit", None)

def predict(dm):
    if best_ntree_limit is not None and best_ntree_limit > 0:
        return booster_final.predict(dm, ntree_limit=best_ntree_limit)
    return booster_final.predict(dm)

def metrics(y, yhat):
    return {
        "R2": float(r2_score(y, yhat)),
        "MAE": float(mean_absolute_error(y, yhat)),
        "RMSE": float(np.sqrt(mean_squared_error(y, yhat))),
    }

final_metrics = {
    "train": metrics(y_train, predict(dtrain)),
    "val":   metrics(y_val,   predict(dval)),
    "test":  metrics(y_test,  predict(dtest)),
}

In [82]:
final_summary = pd.DataFrame(
    [
        {
            "split": k,
            "R2": v["R2"],
            "MAE": v["MAE"],
            "RMSE": v["RMSE"],
        }
        for k, v in final_metrics.items()
    ]
)

final_summary

,split,R2,MAE,RMSE
0,train,0.877047,0.026388,0.035707
1,val,0.768260,0.039763,0.051717
2,test,0.651347,0.042455,0.054902


---

_Jakob Balkovec_